# Word Vector Exercise

In [ ]:
%pip install gensim

In [ ]:
import gensim.downloader as api

print("모델 로딩...")
model = api.load("glove-wiki-gigaword-50")
model

In [ ]:
print("1. 단어 간 유사도 계산")
word1, word2, word3 = "cat", "dog", "car"

print(model.similarity(word1, word2))
print(model.similarity(word1, word3))

In [ ]:
print("2. 비슷한 단어 가져오기")
words = model.most_similar("computer", topn=5)
for word, score in words:
    print(word, score)

In [ ]:
print("3. 가장 안어울리는 것 찾기")
group = ["apple", "banana", "orange", "car"]
print(model.doesnt_match(group))

# Dataset Exercise

In [ ]:
import importlib
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. I am a student."
integers = tokenizer.encode(text)
print(integers)
print(tokenizer.decode(integers))

In [ ]:
import os
import requests

os.makedirs("datas", exist_ok=True)
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "datas/the-verdict.txt"
response = requests.get(url, timeout=30)
response.raise_for_status()
with open(file_path, "wb") as f:
    f.write(response.content)

with open("datas/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[50:250])

In [ ]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

enc_sample = enc_text[50:]
print(len(enc_sample))

context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1 : context_size + 1]

print(x)
print(y)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        print("token_ids 길이", len(token_ids))

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


import tiktoken


def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader


dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False,
)

data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))
print(next(data_iter))

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(
    vocab_size, output_dim
)  # num_embeddings, embedding_dim
print(embedding_layer.weight)

input_ids = torch.tensor([2, 3, 5, 1])
print(embedding_layer(input_ids))

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length,
    shuffle=False,
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)  # batch_size(8), token_cnt(4), embedding_dim(256)

context_length = max_length

pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

input_embedding = (
    token_embeddings + pos_embeddings
)  # 각 Chunk들마다 pos_embeddings 덧셈.
print(input_embedding.shape)

In [ ]:
from transformers import AutoTokenizer

qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
exaone = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-4.0-1.2B")

texts = [
    "안녕하세요, 반갑습니다!",
    "Hello, nice to meet you!",
    "인공지능 기술이 발전하고 있습니다.",
    "Hello! This is an English tokenizer test.",
    "こんにちは！日本語のテストです。",
    "人工智能技术正在快速发展。",
    "The quick brown fox jumps over the lazy dog. 빠른 갈색 여우가 게으른 개를 뛰어넘습니다.",
    "파이썬 프로그래밍은 매우 유용합니다. Python programming is very useful.",
    "123456789 !@#$%^&*() 특수문자",
]

for text in texts:
    print("텍스트", text)

    qwen_tokens = qwen.encode(text, add_special_tokens=True)
    qwen_tokens_strs = [qwen.decode([tok]) for tok in qwen_tokens[:10]]
    print("Qwen", qwen_tokens_strs)

    exaone_tokens = exaone.encode(text, add_special_tokens=True)
    exaone_tokens_strs = [exaone.decode([tok]) for tok in exaone_tokens[:10]]
    print("Exaone", exaone_tokens_strs)

print()
print()

for token_name in ["bos_token", "eos_token", "unk_token", "pad_token"]:
    token = getattr(qwen, token_name, None)
    if token:
        print(token_name, token, getattr(qwen, f"{token_name}_id", None))
    else:
        print(token_name, "없음")

print()
print()

for token_name in ["bos_token", "eos_token", "unk_token", "pad_token"]:
    token = getattr(exaone, token_name, None)
    if token:
        print(token_name, token, getattr(exaone, f"{token_name}_id", None))
    else:
        print(token_name, "없음")

# Basic MLP

In [ ]:
import torch
import torch.nn as nn
import tiktoken

torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
block_size = 8

train_loader = create_dataloader_v1(
    raw_text,
    batch_size=32,
    max_length=block_size,
    stride=1,
    shuffle=False,
)  # Batch 당 32개의 input, target chunk 쌍으로 구성됨.


class MLP(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.fc1 = nn.Linear(block_size * emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, vocab_size)

    def forward(
        self, x
    ):  # x = 입력 2차원 벡터. (batch 내 chunk(block) 갯수, chunk(block) 내 token 갯수)
        e = self.emb(x)  # e = 토큰들을 임베딩값으로 치환함.
        h = e.flatten(
            1
        )  # (B(Chunk 갯수), T(토큰 갯수), C(임베딩 차원)) 에서 T와 C를 평탄화함.
        h = self.fc1(h)  # 선형함수 통과, 가중치 곱셈 후 합 (+bias)
        h = nn.functional.relu(h)  # 활성화 함수 통과.
        return self.fc2(h)  # 출력층 선형함수 통과


model = MLP(vocab_size, block_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for epoch in range(5):
    total = 0
    for input_batch, target_batch in train_loader:
        logits = model(input_batch)
        loss = nn.functional.cross_entropy(logits, target_batch[:, -1])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"epoch {epoch} | loss 평균 {total / len(train_loader): .4f}")


@torch.no_grad()
def generate(model, prompt, max_new_token=30):
    model.eval()
    ids = tokenizer.encode(prompt)
    pad_id = tokenizer.encode(" ")[0]
    for _ in range(max_new_token):
        ctx = ids[-block_size:]
        ctx = [pad_id] * (block_size - len(ctx)) + ctx
        logits = model(torch.tensor([ctx]))
        probs = nn.functional.softmax(
            logits, dim=-1
        )  # logits은 배치사이즈(1) * vocab 사이즈이니, 마지막 vocab 사이즈의 벡터에서 소프트맥스.
        ids.append(torch.multinomial(probs, 1).item())  # probs 중 샘플링 1개
    return tokenizer.decode(ids)


print(generate(model, "I had always thought Jack Gisburn"))

In [ ]:
import torch
import torch.nn as nn
import tiktoken

torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
block_size = 8

train_loader = create_dataloader_v1(
    raw_text,
    batch_size=32,
    max_length=block_size,
    stride=1,
    shuffle=True,
)


class RNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=256):
        super().__init__()
        self.hidden = hidden
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.W_xh = nn.Linear(emb_dim, hidden)
        self.W_hh = nn.Linear(
            hidden, hidden, bias=False
        )  # 이전 은닉 상태를 현재 계산에 가중치와 함께 계산해줌.
        self.fc = nn.Linear(hidden, vocab_size)  # 출력층

    def forward(self, x):
        B, T = x.shape  # 입력 x는 2차원 텐서. [배치사이즈, 블록(청크)사이즈]
        e = self.emb(
            x
        )  # 블록(청크) 내 토큰들을 64차원의 임베딩값으로 바꿈. e는 3차원 텐서. [배치사이즈, 블록(청크)사이즈, 임베딩 차원]
        h = torch.zeros(B, self.hidden)  # 2차원 텐서. [배치사이즈, 은닉층 노드 갯수]

        outs = []
        for t in range(T):
            x_t = e[:, t, :]  # 블록(청크)별 t번째 토큰의 임베딩값
            h = self.W_xh(x_t) + self.W_hh(
                h
            )  # (현재 토큰 * weight + bias) + (이전 은닉상태 * weight)
            h = torch.tanh(h)  # 활성화함수. 현재 은닉상태 계산.
            outs.append(
                h
            )  # 블록(청크) 별 t번째 토큰들의 은닉상태들이 누적됨. outs[0]에는 모든 Chunk들의 0번째 은닉상태 값들이 있음.

        H = torch.stack(outs, dim=1)
        return self.fc(H)


model = RNN(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for epoch in range(5):
    total = 0
    for input_batch, target_batch in train_loader:
        logits = model(
            input_batch
        )  # input_batch에는 여러개(32)의 블록(청크)가 있고, 청크별로 여러개(8)의 토큰이 있고,
        # 토큰별로 여러개(50257)의 확률값이 순전파를 진행하면 나온다.
        loss = nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
        # logits.flatten(0, 1)은 0축부터 1축까지 모두 Flatten 한다. (32*8, 50257) (블록순번이랑 토큰순서를 하나로, 그리고 각 토큰id별로 확률값이 존재.)
        # target_batch.flatten()은 모든 축을 Flatten한다. (32*8,) 블록순번이랑 토큰순서를 하나로, 그리고 토큰id가 존재. (정답표)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    print(f"epoch {epoch} | loss 평균 {total / len(train_loader): .4f}")


@torch.no_grad()
def generate(model, prompt, max_new_token=30):
    model.eval()
    ids = tokenizer.encode(prompt)
    for _ in range(max_new_token):
        ctx = ids[-block_size:]  # 토큰idx 중 뒤에서부터 블록(청크)사이즈만큼 추출
        logits = model(torch.tensor([ctx]))  # 배치 형식으로 넣어줌.
        # logits 결과는 (배치사이즈(1), 블록(청크)사이즈(8), vocab 사이즈) 그리고 각각에 점수값
        logits = logits[:, -1, :]  # 마지막 토큰의 결과만 추출. 차원(축) 축소됨
        probs = nn.functional.softmax(logits, dim=-1) # vocab 별 점수값들에 대해 softmax
        ids.append(torch.multinomial(probs, 1).item())
    return tokenizer.decode(ids)


print(generate(model, "I had always thought Jack Gisburn"))

# Self-Attention

In [ ]:
import torch
import torch.nn as nn


class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)  # dropout = 드랍아웃 확률

        # Causal Mask.
        # register_buffer를 하게 되면, 모델과 함께 cpu 에 적재되거나 GPU로 이동할 수 있음.
        # context_length은 모델이 받을 수 있는 최대 길이. 
        # 참고로 num_tokens은 이것보다 작을 수 있음. (추론시)
        # torch.triu(..., diagonal=1) 하면 대각선 위에가 1인 매트릭스로 나옴.
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = (
            x.shape
        )  # b = 배치사이즈, num_tokens = 블록(청크) 내 토큰 갯수, d_in = 입력 차원

        queries = self.W_query(
            x
        )  # W_query의 입력차원은 d_in, 출력차원은 d_out이다. 따라서 queries의 차원은 [b, num_tokens, d_out]
        keys = self.W_key(
            x
        )  # W_key의 입력차원은 d_in, 출력차원은 d_out이다. 따라서 keys의 차원은 [b, num_tokens, d_out]
        values = self.W_value(x)

        attention_scores = queries @ keys.transpose(
            1, 2
        )  # queries 및 keys 둘다 [b, num_tokens, d_out] 이다.
        # 따라서 내적을 위해서 keys의 num_tokens의 축(1)과 d_out의 축(2)을 바꾸고 내적한다.
        # attention_score 는 [b, num_tokens, num_tokens] 3차원 텐서가 된다.
        # 블록(청크)별로 두 토큰 임베딩*weight(q, k) 끼리의 유사도를 모두 계산한것이다.
        # (순서 영향 있음. A,B와의 관계와 B,A 관계는 다름.)

        # 마스크
        # masked_fill_(..., -torch.inf) 은 마스크의 값이 1(True)인 곳은 모두 -torch.inf로 치환하라는 뜻
        attention_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        dk = keys.shape[-1]  # d_out 값
        attention_weights = torch.softmax(attention_scores / dk**0.5, dim=-1)
        # 일단 attention_score들을 스케일을 맞추기 위해서 dk**0.5 으로 나눔.
        # attention_scores는 b, num_tokens, num_tokens 인데, 마지막 num_tokens(dim=-1)의 값을 이용하여 확률값 구함.
        # 첫번째 행(단어)의 경우엔, 자기 자신과의 유사도밖에 없음.
        # 두번째 행(단어)의 경우엔, 첫번째 단어와의 유사도, 자기 자신과의 유사도 값을 이용하여 확률값 구함.

        # 오버피팅을 방지하기 위한 드랍아웃
        attention_weights = self.dropout(attention_weights)

        # 어텐션 가중치를 values에 내적.
        # 어텐션 가중치 [b, num_tokens, num_tokens]
        # values [b, num_tokens, d_out]
        context_vec = attention_weights @ values
        # context_vec [b, num_tokens, d_out]

        # 최종적으로 배치*토큰별 문맥 벡터 값들이 나옴.
        return context_vec
    
import torch
  
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[1]
d_out = 2

batch = torch.stack((inputs, inputs), dim=0)
# 동일한 배치 2짜리, [b, t, e] 3차원 텐서로 만든다.
# batch = inputs.unsqueeze(0)
# batch = torch.stack((inputs, ), dim=0)
# batch = torch.stack([inputs], dim=0)

torch.manual_seed(123)

context_length = batch.shape[1]

print("context length: ", context_length)

ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

# Multi-head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        # x는 [b, num_tokens, d_in]. queries는 [b, num_tokens, d_out]
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # 원래 queries는 [b, num_tokens, d_out]인데, 뒤 d_out을 헤드 갯수 * 헤드별 차원으로 나눈다.
        # queries는 [b, num_tokens, num_heads, head_dim] 으로 바뀐다.
        # 여러개의 W_query를 선언해서 멀티헤드를 만드는게 아니라, 하나의 W_query에 멀티헤드들의 Weight를 한번에 담음.
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # [b, num_tokens, num_heads, head_dim] 에서 [b, num_heads, num_tokens, head_dim] 으로 전환
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # keys의 num_tokens과 head_dim의 축을 바꿈
        # attn_score는 이제 [b, num_heads, num_tokens, num_tokens]
        attn_score = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_score = attn_score.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_score / keys.shape[-1] ** 0.5, dim=-1)
        # keys.shape[-1]는 head_dim
        # dim=-1은 유사도 값들을 들고있는 [num_tokens] 벡터를 이용하여 softmax 하겠단 의미.
        attn_weights = self.dropout(attn_weights)

        # attn_weights는 [b, num_heads, num_tokens, num_tokens]
        # values는 [b, num_heads, num_tokens, head_dim]
        # attn_weights @ values 는 [b, num_heads, num_tokens, head_dim]
        # context_vecs 는 [b, num_tokens, num_heads, head_dim]
        context_vecs = (attn_weights @ values).transpose(1, 2)

        # num_heads, head_dim 쪽을 concat 한다.
        context_vecs = context_vecs.contiguous().view(b, num_tokens, self.d_out)
        context_vecs = self.out_proj(context_vecs)  # output projection weight

        return context_vecs


torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(
    d_in, d_out, context_length=context_length, dropout=0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

# GPT

In [ ]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]

            # Chunk 안에는 Token Index값들이 들어있다.
            # input_ids 에는 여러개의 Token Index 값을 가지고 있는 여러개의 Chunk로 구성되어 있다.
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]


def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.output_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)  # [b, num_tokens, d_out]
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        queries = queries.transpose(1, 2)  # [b, num_heads, num_tokens, head_dim]
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_score = queries @ keys.transpose(2, 3)
        # [b, num_heads, num_tokens, num_tokens]
        attn_score.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        attn_weight = attn_score / self.head_dim**0.5
        attn_weight = torch.softmax(attn_weight, dim=-1)
        attn_weight = self.dropout(attn_weight)

        context_vecs = attn_weight @ values  # [b, num_head, num_tokens, head_dim]
        context_vecs = context_vecs.transpose(1, 2)
        # [b, num_tokens, num_head, head_dim]

        context_vecs = context_vecs.contiguous().view(b, num_tokens, self.d_out)
        # [b, num_tokens, d_out]
        context_vecs = self.output_proj(context_vecs)
        # [b, num_tokens, d_out]

        return context_vecs

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  # 분모가 0이 되는것을 방지하는 앱실론값
        self.scale = nn.Parameter(torch.ones(emb_dim))  # Gamma
        self.shift = nn.Parameter(torch.zeros(emb_dim))  # Beta

    def forward(self, x):  # [b, num_tokens, emb_dim]
        mean = x.mean(dim=-1, keepdim=True)
        # [b, num_tokens, 1]
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return (
            0.5
            * x
            * (
                1
                + nn.functional.tanh(
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # GPT 구조에서 은측층 크기를 4배로 확장하고, 확장/축소 은닉층 사이에 GELU 활성화함수를 추가한다.
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], cfg["emb_dim"] * 4),
            GELU(),
            nn.Linear(cfg["emb_dim"] * 4, cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            cfg["emb_dim"],
            cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # norm 및 attention 이후 기울기 소실 방지

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        tok_embdes = self.tok_emb(in_idx)
        pos_embdes = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        x = tok_embdes + pos_embdes
        x = self.dropout(x)

        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits


def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # 단어 집합 크기
    "context_length": 1024,  # 최대 문맥 길이
    "emb_dim": 768,  # 임베딩 차원
    "n_heads": 12,  # 어텐션 헤드 수
    "n_layers": 12,  # 레이어 수
    "drop_rate": 0.1,  # 드롭아웃 비율
    "qkv_bias": False,  # Q,K,V 편향 사용 여부
}

torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.eval()

start_context = "Hello, I am"

tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"],
)

decoded_text = tokenizer.decode(out.squeeze(0).tolist())

print(decoded_text)

# 모델 훈련

In [ ]:
import matplotlib.pyplot as plt
import os
import requests
import torch
import tiktoken


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)

    # input_batch [b, num_tokens]
    logits = model(input_batch)
    # logits [b, num_tokens, vocab_size]
    # target_batch [b, num_tokens]
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss


def train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs,
    eval_freq,
    eval_iter,
    start_context,
    tokenizer,
):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen = 0
    global_step = -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(
                    f"{epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}"
                )

        generate_and_print_sample(model, tokenizer, device, start_context)
    return train_losses, val_losses, track_tokens_seen


def generate(model, idx, max_new_tokens, context_size, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)
    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()

    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)

    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)

    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded, max_new_tokens=50, context_size=context_size
        )
        decoded_text = token_ids_to_text(token_ids, tokenizer)
        print(decoded_text.replace("\n", " "))
    model.train()


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots()

    # 기본 X축: Epoch 기준
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")

    # 보조 X축 (상단): 처리한 토큰 수(Tokens seen) 기준
    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)  # 투명한 그래프로 축만 생성
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()
    plt.show()

In [ ]:
def main(gpt_config, settings):
    torch.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    file_path = "datas/the-verdict.txt"
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    model = GPTModel(gpt_config)
    model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=settings["learning_rate"],
        weight_decay=settings["weight_decay"],
    )

    train_ratio = 0.90
    split_idx = int(train_ratio * len(text_data))

    train_loader = create_dataloader_v1(
        text_data[:split_idx],
        batch_size=settings["batch_size"],
        max_length=gpt_config["context_length"],
        stride=gpt_config["context_length"],
        drop_last=True,
        shuffle=True,
        num_workers=0,
    )

    val_loader = create_dataloader_v1(
        text_data[split_idx:],
        batch_size=settings["batch_size"],
        max_length=gpt_config["context_length"],
        stride=gpt_config["context_length"],
        drop_last=False,
        shuffle=False,
        num_workers=0,
    )

    tokenizer = tiktoken.get_encoding("gpt2")
    train_losses, val_losses, tokens_seen = train_model_simple(
        model,
        train_loader,
        val_loader,
        optimizer,
        device,
        num_epochs=settings["num_epochs"],
        eval_freq=5,
        eval_iter=1,
        start_context="Every effort moves you",
        tokenizer=tokenizer,
    )

    return train_losses, val_losses, tokens_seen, model

In [ ]:
# GPT-2 Small (124M) 모델 설정
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # 어휘 크기
    "context_length": 256,  # 훈련 속도를 위해 원본(1024)보다 줄임
    "emb_dim": 768,  # 임베딩 벡터 차원
    "n_heads": 12,  # 어텐션 헤드 개수
    "n_layers": 12,  # 레이어 깊이
    "drop_rate": 0.1,  # 과적합 방지 드롭아웃
    "qkv_bias": False,
}

# 학습 하이퍼파라미터
OTHER_SETTINGS = {
    "learning_rate": 5e-4,
    "num_epochs": 10,
    "batch_size": 2,
    "weight_decay": 0.1,
}

###########################
# 메인 실행: 훈련
###########################
train_losses, val_losses, tokens_seen, model = main(GPT_CONFIG_124M, OTHER_SETTINGS)

###########################
# 훈련 결과 시각화 및 저장
###########################

# 에포크 축 생성 (0부터 num_epochs까지)
epochs_tensor = torch.linspace(0, OTHER_SETTINGS["num_epochs"], len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/loss.pdf")

# 모델 저장 및 로드 테스트
torch.save(model.state_dict(), "outputs/model.pth")

# 디바이스 설정 (모델 로드 전에 정의 필요)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 다시 불러오기 테스트
model = GPTModel(GPT_CONFIG_124M)  # 1. 빈 모델 생성 (현재 CPU에 있음)
model.load_state_dict(
    torch.load("outputs/model.pth", weights_only=True)
)  # 2. 가중치 로드
model.to(device)  # <--- [핵심 수정] 3. 모델을 GPU로 이동시켜야 함

torch.manual_seed(123)
###########################
# 생성 테스트 (Inference)
###########################
tokenizer = tiktoken.get_encoding("gpt2")

# 훈련된 모델로 텍스트 생성 시도
# Temperature=1.4로 설정하여 다소 창의적/다양한 문장 생성 유도
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))